In [123]:
import pandas as pd

import os
path2data = './raw_data/NL to LTL Synthetic Dataset'
data = os.listdir(path2data)
data = [x for x in data if (  ("train" in x and 'unrestricted' in x and "~" not in x) )]
data

['unrestricted_train_dataset-50.csv', 'unrestricted_train_dataset-140.csv']

In [124]:
import spot
def simplify_formula(formula_str):

    f = spot.formula(formula_str)
    return f


def extract_atomic_propositions(formula_str):
    f = spot.formula(formula_str)
    aps = spot.atomic_prop_collect(f)
    
    mapping = []
    for i, atom in enumerate(aps):
        mapping.append(str(atom))
        

    return mapping


In [125]:
lista = []
for elem in data:
    df = pd.read_csv(path2data + '/' + elem)[['ltl','en']]
    df['source'] = str(elem)
    lista.append( df )
result = pd.concat(lista, ignore_index=True)
result

,ltl,en,source
0,G(!( brake_is_pressed | house_is_open | train_...,under no circumstances either the brake is pre...,unrestricted_train_dataset-50.csv
1,G(!( car_starts | semaphore_is_red | bar_is_up )),"under no condition either a car starts, a sema...",unrestricted_train_dataset-50.csv
2,G(!( escalator_speeds_up & train_derails & hou...,"under no condition the escalator speeds up, th...",unrestricted_train_dataset-50.csv
3,G(!( constructor_instantiate_objects & bar_is_...,it is never the case that a constructor instan...,unrestricted_train_dataset-50.csv
4,G(!( table_has_been_moved & table_is_brown & m...,it never happens that the table has been moved...,unrestricted_train_dataset-50.csv
...,...,...,...
189995,G(( elevator_rises ) -> G(( sensor_captures_da...,each time the elevator rises then always when ...,unrestricted_train_dataset-140.csv
189996,G(( train_derails ) -> G(( escalator_moves ) -...,when the train derails then it will happen tha...,unrestricted_train_dataset-140.csv
189997,G(( brake_is_pressed ) -> G(( constructor_inst...,the brake is pressed involves that whenever th...,unrestricted_train_dataset-140.csv
189998,G(( elevator_is_blocked ) -> G(( sensor_captur...,when the elevator is blocked then if the senso...,unrestricted_train_dataset-140.csv


In [126]:
result = result.rename(columns={'ltl': 'original LTL', 'en': 'original NL'})

result

,original LTL,original NL,source
0,G(!( brake_is_pressed | house_is_open | train_...,under no circumstances either the brake is pre...,unrestricted_train_dataset-50.csv
1,G(!( car_starts | semaphore_is_red | bar_is_up )),"under no condition either a car starts, a sema...",unrestricted_train_dataset-50.csv
2,G(!( escalator_speeds_up & train_derails & hou...,"under no condition the escalator speeds up, th...",unrestricted_train_dataset-50.csv
3,G(!( constructor_instantiate_objects & bar_is_...,it is never the case that a constructor instan...,unrestricted_train_dataset-50.csv
4,G(!( table_has_been_moved & table_is_brown & m...,it never happens that the table has been moved...,unrestricted_train_dataset-50.csv
...,...,...,...
189995,G(( elevator_rises ) -> G(( sensor_captures_da...,each time the elevator rises then always when ...,unrestricted_train_dataset-140.csv
189996,G(( train_derails ) -> G(( escalator_moves ) -...,when the train derails then it will happen tha...,unrestricted_train_dataset-140.csv
189997,G(( brake_is_pressed ) -> G(( constructor_inst...,the brake is pressed involves that whenever th...,unrestricted_train_dataset-140.csv
189998,G(( elevator_is_blocked ) -> G(( sensor_captur...,when the elevator is blocked then if the senso...,unrestricted_train_dataset-140.csv


In [127]:

result['Spot LTL'] = result['original LTL'].apply(lambda x: str(simplify_formula(x)))
result['APs'] = result['original LTL'].apply(lambda x: extract_atomic_propositions(x))
result

,original LTL,original NL,source,Spot LTL,APs
0,G(!( brake_is_pressed | house_is_open | train_...,under no circumstances either the brake is pre...,unrestricted_train_dataset-50.csv,G!(brake_is_pressed | house_is_open | train_is...,"[brake_is_pressed, house_is_open, train_is_cro..."
1,G(!( car_starts | semaphore_is_red | bar_is_up )),"under no condition either a car starts, a sema...",unrestricted_train_dataset-50.csv,G!(bar_is_up | car_starts | semaphore_is_red),"[car_starts, semaphore_is_red, bar_is_up]"
2,G(!( escalator_speeds_up & train_derails & hou...,"under no condition the escalator speeds up, th...",unrestricted_train_dataset-50.csv,G!(escalator_speeds_up & house_is_built & trai...,"[escalator_speeds_up, train_derails, house_is_..."
3,G(!( constructor_instantiate_objects & bar_is_...,it is never the case that a constructor instan...,unrestricted_train_dataset-50.csv,G!(bar_is_closing & constructor_instantiate_ob...,"[constructor_instantiate_objects, bar_is_closi..."
4,G(!( table_has_been_moved & table_is_brown & m...,it never happens that the table has been moved...,unrestricted_train_dataset-50.csv,G!(manager_collect_claims & table_has_been_mov...,"[table_has_been_moved, table_is_brown, manager..."
...,...,...,...,...,...
189995,G(( elevator_rises ) -> G(( sensor_captures_da...,each time the elevator rises then always when ...,unrestricted_train_dataset-140.csv,G(elevator_rises -> G(sensor_captures_data -> ...,"[train_is_crossing, sensor_captures_data, elev..."
189996,G(( train_derails ) -> G(( escalator_moves ) -...,when the train derails then it will happen tha...,unrestricted_train_dataset-140.csv,G(train_derails -> G(escalator_moves -> Feleva...,"[train_derails, escalator_moves, elevator_is_b..."
189997,G(( brake_is_pressed ) -> G(( constructor_inst...,the brake is pressed involves that whenever th...,unrestricted_train_dataset-140.csv,G(brake_is_pressed -> G(constructor_instantiat...,"[brake_is_pressed, constructor_instantiate_obj..."
189998,G(( elevator_is_blocked ) -> G(( sensor_captur...,when the elevator is blocked then if the senso...,unrestricted_train_dataset-140.csv,G(elevator_is_blocked -> G(sensor_captures_dat...,"[sensor_captures_data, elevator_is_blocked, tr..."


In [128]:
import re
def swap_row_propositions(row, prop_col, text_col):
    propositions = row[prop_col]
    text = row[text_col]
    
    if not isinstance(propositions, list) or not isinstance(text, str):
        return text

    # 1. Create mapping for THIS row: {'car starts': 'car_starts'}
    mapping = {p.replace('_', ' '): p for p in propositions}
    
    # 2. Build a regex pattern from these specific keys
    # Sort by length descending to match longest phrases first
    sorted_keys = sorted(mapping.keys(), key=len, reverse=True)
    pattern = re.compile(r'\b(' + '|'.join(map(re.escape, sorted_keys)) + r')\b')
    
    # 3. Replace matches in the text using the mapping
    return pattern.sub(lambda m: mapping[m.group(0)], text)

# Apply the function across the rows
result['ap_NL'] = result.apply(
    lambda row: swap_row_propositions(row, 'APs', 'original NL'), 
    axis=1
)

In [129]:
result

,original LTL,original NL,source,Spot LTL,APs,ap_NL
0,G(!( brake_is_pressed | house_is_open | train_...,under no circumstances either the brake is pre...,unrestricted_train_dataset-50.csv,G!(brake_is_pressed | house_is_open | train_is...,"[brake_is_pressed, house_is_open, train_is_cro...",under no circumstances either the brake_is_pre...
1,G(!( car_starts | semaphore_is_red | bar_is_up )),"under no condition either a car starts, a sema...",unrestricted_train_dataset-50.csv,G!(bar_is_up | car_starts | semaphore_is_red),"[car_starts, semaphore_is_red, bar_is_up]","under no condition either a car_starts, a sema..."
2,G(!( escalator_speeds_up & train_derails & hou...,"under no condition the escalator speeds up, th...",unrestricted_train_dataset-50.csv,G!(escalator_speeds_up & house_is_built & trai...,"[escalator_speeds_up, train_derails, house_is_...","under no condition the escalator_speeds_up, th..."
3,G(!( constructor_instantiate_objects & bar_is_...,it is never the case that a constructor instan...,unrestricted_train_dataset-50.csv,G!(bar_is_closing & constructor_instantiate_ob...,"[constructor_instantiate_objects, bar_is_closi...",it is never the case that a constructor_instan...
4,G(!( table_has_been_moved & table_is_brown & m...,it never happens that the table has been moved...,unrestricted_train_dataset-50.csv,G!(manager_collect_claims & table_has_been_mov...,"[table_has_been_moved, table_is_brown, manager...",it never happens that the table_has_been_moved...
...,...,...,...,...,...,...
189995,G(( elevator_rises ) -> G(( sensor_captures_da...,each time the elevator rises then always when ...,unrestricted_train_dataset-140.csv,G(elevator_rises -> G(sensor_captures_data -> ...,"[train_is_crossing, sensor_captures_data, elev...",each time the elevator_rises then always when ...
189996,G(( train_derails ) -> G(( escalator_moves ) -...,when the train derails then it will happen tha...,unrestricted_train_dataset-140.csv,G(train_derails -> G(escalator_moves -> Feleva...,"[train_derails, escalator_moves, elevator_is_b...",when the train_derails then it will happen tha...
189997,G(( brake_is_pressed ) -> G(( constructor_inst...,the brake is pressed involves that whenever th...,unrestricted_train_dataset-140.csv,G(brake_is_pressed -> G(constructor_instantiat...,"[brake_is_pressed, constructor_instantiate_obj...",the brake_is_pressed involves that whenever th...
189998,G(( elevator_is_blocked ) -> G(( sensor_captur...,when the elevator is blocked then if the senso...,unrestricted_train_dataset-140.csv,G(elevator_is_blocked -> G(sensor_captures_dat...,"[sensor_captures_data, elevator_is_blocked, tr...",when the elevator_is_blocked then if the senso...


In [130]:
result.to_csv('./processed_data/NL to LTL Synthetic Dataset/processed_data.csv', index=False, sep=';')

In [172]:
def english_to_ltl(s):
    s = s.replace('globally', 'G')
    s = s.replace('finally', 'F')
    s = s.replace('next', 'X')
    s = s.replace('or', '|')
    s = s.replace('double_implies', '<->')
    s = s.replace('implies', '->')
    s = s.replace('and', '&')
    s = s.replace('not', '!')
    s = s.replace('negation', '!')
    s = s.replace('imply', '->')
    s = s.replace('until', 'U')
    return s

eng_to_ltl_dict = {'globally': 'G',
'finally': 'F',
'next': 'X',
'or': '|',
'double_implies': '<->',
'implies': '->',
'and': '&',
'not': '!',
'negation': '!',
'imply': '->',
'until': 'U'}


In [166]:
path2data = './raw_data/VLTL-Bench/new/'
data = os.listdir(path2data)
data = [x for x in data ]
data

['search_and_rescue.jsonl', 'traffic_light.jsonl', 'warehouse.jsonl']

In [175]:
df = {}
for file in data:
    df[file] = pd.read_json(os.path.join(path2data, file), lines=True)

reserved = ['globally', 'finally', 'next', 'or', 'double_implies', 
            'implies', 'and', 'until', 'not', 'negation', 'imply', '(', ')']

def wrap_propositions(token_list):
    return [f'"{item}"' if item not in reserved else item for item in token_list]

def swap_oprerations(token_list):
    return [f'{eng_to_ltl_dict[item]}' if item in eng_to_ltl_dict else item for item in token_list]

for file in df:
    df[file]['source'] = file
    df[file]['original NL'] = df[file]['sentence'].str.join(' ')
    df[file]['tl'] = df[file]['tl'].apply(wrap_propositions)
    df[file]['tl'] = df[file]['tl'].apply(swap_oprerations)
    df[file]['original LTL'] = df[file]['tl'].str.join(' ').replace('""', '"', regex=True)
    df[file]['Spot LTL'] = df[file]['original LTL'].apply(english_to_ltl)

def get_flat_map(nested_dict):
    # We combine all args_canon -> args_ref pairs into one dict
    if not isinstance(nested_dict, dict):
        return {}
    
    flat_map = {}
    for prop in nested_dict.values():
        # Zip canon keys to ref values and update our master map
        flat_map.update(zip(prop.get('args_ref', []), prop.get('args_canon', [])))
    return flat_map

for file in df:
    df[file]['AP_dict'] = df[file]['prop_dict'].apply(lambda x: {x[y]['action_ref']: x[y]['action_canon'] for y in x})
    
    df[file]['AP_misc_dict'] = df[file]['prop_dict'].apply(get_flat_map)
    df[file]['merged_dict'] = df[file].apply(lambda row: row['AP_misc_dict'] | row['AP_dict'], axis=1)

def replace_from_mapping(row, text_column):
    text = str(row[text_column])
    mapping = row['merged_dict']
    
    # Iterate through the dictionary for this specific row
    for old_val, new_val in mapping.items():
        # replace() handles all instances of that key in the string
        text = text.replace(old_val, new_val)
    
    return text

import csv

for file in df:
    # Apply the function to your text column
    # Replace 'my_text_col' with the actual name of your string column
    df[file]['original NL'] = df[file].apply(replace_from_mapping, axis=1, text_column='original NL')
    df[file]['Spot LTL'] = df[file]['Spot LTL'].apply(lambda x: str(simplify_formula(x)))
    df[file]['APs'] = df[file]['Spot LTL'].apply(lambda x: extract_atomic_propositions(x))
    df[file][['source','APs','original NL','original LTL','Spot LTL']].to_csv('./processed_data/VLTL-Bench/new/'+file.split('.')[0]+'.csv', index=False, sep=';',index_label=False, quoting=csv.QUOTE_MINIMAL)
#

In [176]:
path2data = './raw_data/VLTL-Bench/prev/'
data = os.listdir(path2data)
data = [x for x in data ]
data

['GLTL.jsonl', 'navi.jsonl', 'conformal.jsonl', 'cleanup_world.jsonl']

In [177]:
df = {}
for file in data:
    df[file] = pd.read_json(os.path.join(path2data, file), lines=True)

reserved = ['globally', 'finally', 'next', 'or', 'double_implies', 
            'implies', 'and', 'until', 'not', 'negation', 'imply', '(', ')']

def wrap_propositions(token_list):
    return [f'"{item}"' if item not in reserved else item for item in token_list]


for file in df:
    df[file]['source'] = file
    df[file]['original NL'] = df[file]['sentence'].str.join(' ')
    df[file]['tl'] = df[file]['tl'].apply(wrap_propositions)
    df[file]['original LTL'] = df[file]['tl'].str.join(' ').replace('""', '"', regex=True)
    df[file]['Spot LTL'] = df[file]['original LTL'].apply(english_to_ltl)

def get_flat_map(nested_dict):
    # We combine all args_canon -> args_ref pairs into one dict
    if not isinstance(nested_dict, dict):
        return {}
    
    flat_map = {}
    for prop in nested_dict.values():
        # Zip canon keys to ref values and update our master map
        flat_map.update(zip(prop.get('args_ref', []), prop.get('args_canon', [])))
    return flat_map

for file in df:
    df[file]['AP_dict'] = df[file]['prop_dict'].apply(lambda x: {x[y]['action_ref']: x[y]['action_canon'] for y in x})
    
    df[file]['AP_misc_dict'] = df[file]['prop_dict'].apply(get_flat_map)
    df[file]['merged_dict'] = df[file].apply(lambda row: row['AP_misc_dict'] | row['AP_dict'], axis=1)

def replace_from_mapping(row, text_column):
    text = str(row[text_column])
    mapping = row['merged_dict']
    
    # Iterate through the dictionary for this specific row
    for old_val, new_val in mapping.items():
        # replace() handles all instances of that key in the string
        text = text.replace(old_val, new_val)
    
    return text

import csv

for file in df:
    # Apply the function to your text column
    # Replace 'my_text_col' with the actual name of your string column
    df[file]['original NL'] = df[file].apply(replace_from_mapping, axis=1, text_column='original NL')
    df[file]['Spot LTL'] = df[file]['Spot LTL'].apply(lambda x: str(simplify_formula(x)))
    df[file]['APs'] = df[file]['Spot LTL'].apply(lambda x: extract_atomic_propositions(x))
    df[file][['source','APs','original NL','original LTL','Spot LTL']].to_csv('./processed_data/VLTL-Bench/prev/'+file.split('.')[0]+'.csv', index=False, sep=';',index_label=False, quoting=csv.QUOTE_MINIMAL)
#

In [187]:
import json
from pprint import pprint

with open('./raw_data/ConformalNL2LTL/calibration.json') as json_data:
    d = json.load(json_data)
    json_data.close()

for ind in range(len(d)):
    d[ind]['ltlequ'] = d[ind]['ltlequ'].strip("/")
    if "U" in d[ind]['ltlequ']:
       d[ind]['ltlequ'] = d[ind]['ltlequ'].replace("U", " U ")
    
    d[ind]['Spot LTL'] = str(simplify_formula(str(d[ind]['ltlequ'])))

d


[{'nlTask': 'Take a picture of building 2',
  'ltlequ': '<>(building_2&&photo)',
  'Spot LTL': 'F(building_2 & photo)'},
 {'nlTask': 'Take bottle 3 from store 1 and put it in hospital 2',
  'ltlequ': '<>(shop_1&&<>(p_bottle_3&&<>(hospital_2&&pd)))',
  'Spot LTL': 'F(shop_1 & F(p_bottle_3 & F(hospital_2 & pd)))'},
 {'nlTask': 'Go to parking lot 2 and pick up dish 2',
  'ltlequ': '<>(parking_lot_2&&p_dish_2)',
  'Spot LTL': 'F(p_dish_2 & parking_lot_2)'},
 {'nlTask': 'Go to hospital 3 and pick up water bottle 4 while never entering pharmacy 2',
  'ltlequ': '<>(hospital_3&&p_bottle_4)&&[]!pharmacy_2',
  'Spot LTL': 'F(hospital_3 & p_bottle_4) & G!pharmacy_2'},
 {'nlTask': 'Stay in parking lot 4 until you reach car 5',
  'ltlequ': 'parking_lot_4 U car_5',
  'Spot LTL': 'parking_lot_4 U car_5'},
 {'nlTask': 'Do not enter street 5 until you reach store 1 as you pick up package 4 from store 1',
  'ltlequ': '!street_5 U store_1&&<>(store_1&&p_package_4)',
  'Spot LTL': '(!street_5 U store_1) &

In [190]:
df = pd.json_normalize(d).rename(columns={'ltlequ': 'original LTL', 'nlTask': 'original NL'})
df['APs'] = df['Spot LTL'].apply(lambda x: extract_atomic_propositions(x))
df['ap_NL'] = df.apply(
    lambda row: swap_row_propositions(row, 'APs', 'original NL'), 
    axis=1
)
df['dict_AP']=df['APs'].apply(lambda x: {x.removeprefix('p_').replace('_', ' ') : x for x in x})

def dynamic_replace(row):
    text = row['ap_NL']
    mapping = row['dict_AP']
    
    if not mapping or not isinstance(mapping, dict):
        return text
    
    # Create a regex pattern from the keys of the row's dictionary
    # \b ensures we only match whole words
    pattern = re.compile(r'\b(' + '|'.join(re.escape(k) for k in mapping.keys()) + r')\b')
    
    # Replace using the dictionary values
    return pattern.sub(lambda m: mapping[m.group(0)], text)

df['ap_NL'] = df.apply(dynamic_replace, axis=1)
df.to_csv('./processed_data/ConformalNL2LTL/processed_data.csv', index=False, sep=';')
df

,original NL,original LTL,Spot LTL,APs,ap_NL,dict_AP
0,Take a picture of building 2,<>(building_2&&photo),F(building_2 & photo),"[photo, building_2]",Take a picture of building_2,"{'photo': 'photo', 'building 2': 'building_2'}"
1,Take bottle 3 from store 1 and put it in hospi...,<>(shop_1&&<>(p_bottle_3&&<>(hospital_2&&pd))),F(shop_1 & F(p_bottle_3 & F(hospital_2 & pd))),"[pd, p_bottle_3, shop_1, hospital_2]",Take p_bottle_3 from store 1 and put it in hos...,"{'pd': 'pd', 'bottle 3': 'p_bottle_3', 'shop 1..."
2,Go to parking lot 2 and pick up dish 2,<>(parking_lot_2&&p_dish_2),F(p_dish_2 & parking_lot_2),"[p_dish_2, parking_lot_2]",Go to parking_lot_2 and pick up p_dish_2,"{'dish 2': 'p_dish_2', 'parking lot 2': 'parki..."
3,Go to hospital 3 and pick up water bottle 4 wh...,<>(hospital_3&&p_bottle_4)&&[]!pharmacy_2,F(hospital_3 & p_bottle_4) & G!pharmacy_2,"[p_bottle_4, hospital_3, pharmacy_2]",Go to hospital_3 and pick up water p_bottle_4 ...,"{'bottle 4': 'p_bottle_4', 'hospital 3': 'hosp..."
4,Stay in parking lot 4 until you reach car 5,parking_lot_4 U car_5,parking_lot_4 U car_5,"[parking_lot_4, car_5]",Stay in parking_lot_4 until you reach car_5,"{'parking lot 4': 'parking_lot_4', 'car 5': 'c..."
...,...,...,...,...,...,...
995,Persistently taking crate 6 to building 5 afte...,[]<>(store_2&&<>(p_crate_6&&<>(building_5&&pd))),GF(store_2 & F(p_crate_6 & F(building_5 & pd))),"[pd, building_5, p_crate_6, store_2]",Persistently taking p_crate_6 to building_5 af...,"{'pd': 'pd', 'building 5': 'building_5', 'crat..."
996,"Deliver box 1 from store 6 to house 9, then pr...",<>(store_6&&<>(p_box_1&&<>(house_9&&pd)))&&<>(...,F(store_6 & F(p_box_1 & F(house_9 & pd))) & F(...,"[pd, house_9, intersection_5, p_box_1, store_6]","Deliver p_box_1 from store_6 to house_9, then ...","{'pd': 'pd', 'house 9': 'house_9', 'intersecti..."
997,Ensure you transport box 1 from store 6 to hou...,<>(store_6&&<>(p_box_1&&<>(house_9&&pd)))&&<>(...,F(store_6 & F(p_box_1 & F(house_9 & pd))) & F(...,"[pd, house_9, intersection_5, p_box_1, store_6]",Ensure you transport p_box_1 from store_6 to h...,"{'pd': 'pd', 'house 9': 'house_9', 'intersecti..."
998,"Move box 1 from store 6 to house 9, and then g...",<>(store_6&&<>(p_box_1&&<>(house_9&&pd)))&&<>(...,F(store_6 & F(p_box_1 & F(house_9 & pd))) & F(...,"[pd, house_9, intersection_5, p_box_1, store_6]","Move p_box_1 from store_6 to house_9, and then...","{'pd': 'pd', 'house 9': 'house_9', 'intersecti..."
